In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("AmesHousing.csv")

# First five rows
print("First 5 rows:")
print(df.head())

# Column data types
print("\nColumn data types:")
print(df.dtypes)

# DataFrame shape
print(f"\nDataFrame shape: {df.shape[0]} rows x {df.shape[1]} columns")

In [ ]:
# Null value count and percentage per column
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / df.shape[0]) * 100

null_summary = pd.DataFrame({
    "null_count": null_counts,
    "null_pct": null_pct.round(2)
}).sort_values("null_pct", ascending=False)

print("Null count / percentage per column:")
print(null_summary)

# Columns exceeding 20% null rate
high_null_cols = null_summary[null_summary["null_pct"] > 20].index.tolist()
print(f"\nColumns with > 20% nulls: {high_null_cols}")

# For columns below 20% nulls, fill NUMERIC columns with the median
numeric_cols = df.select_dtypes(include="number").columns.tolist()
low_null_numeric = [c for c in numeric_cols if null_pct[c] < 20]

print(f"\nNumeric columns with <20% nulls (median-imputed): {low_null_numeric}")

for col in low_null_numeric:
    df[col] = df[col].fillna(df[col].median())

print("\nRemaining nulls in those columns after median fill:")
print(df[low_null_numeric].isnull().sum())

In [ ]:
# Count duplicate rows
dup_count = df.duplicated().sum()
print(f"Duplicate rows found: {dup_count}")

# Capture null % before removing duplicates (to compare after)
null_pct_before_dedup = (df.isnull().sum() / df.shape[0] * 100).round(2)

# Remove duplicates
df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows removed: {dup_count}. New shape: {df.shape}")

# Compare null % before vs after
null_pct_after_dedup = (df.isnull().sum() / df.shape[0] * 100).round(2)
null_pct_change = (null_pct_after_dedup - null_pct_before_dedup).round(3)

print("\nChange in null % per column after removing duplicates (non-zero changes only):")
change_table = pd.DataFrame({
    "before_%": null_pct_before_dedup,
    "after_%": null_pct_after_dedup,
    "change": null_pct_change
})
print(change_table[change_table["change"] != 0])
if change_table["change"].eq(0).all():
    print("(no changes - identical null % for every column)")

In [ ]:
# Total memory usage BEFORE any dtype correction
mem_before = df.memory_usage(deep=True).sum()
print(f"Total memory usage BEFORE dtype correction: {mem_before / 1024:.2f} KB")

# Check a numeric-looking column that's actually stored as text/object
print("\nMS SubClass dtype:", df["MS SubClass"].dtype, "- sample values:", df["MS SubClass"].unique()[:10])
print("Overall Qual dtype:", df["Overall Qual"].dtype, "- sample values:", df["Overall Qual"].unique()[:10])

# MS SubClass is a building-class CODE (numeric-looking but actually categorical -
# e.g. 20 = "1-story 1946+", 60 = "2-story 1946+"), so treat it as category, not a number
df["MS SubClass"] = df["MS SubClass"].astype(str).astype("category")

# Neighborhood is a highly repetitive string column - good candidate for category dtype
mem_before_cat = df.memory_usage(deep=True).sum()
print(f"\nMemory usage BEFORE converting 'Neighborhood' to category: {mem_before_cat / 1024:.2f} KB")
print(f"'Neighborhood' unique values: {df['Neighborhood'].nunique()}")

df["Neighborhood"] = df["Neighborhood"].astype("category")

mem_after_cat = df.memory_usage(deep=True).sum()
print(f"Memory usage AFTER converting 'Neighborhood' to category: {mem_after_cat / 1024:.2f} KB")
print(f"Memory saved: {(mem_before_cat - mem_after_cat) / 1024:.2f} KB "
      f"({(1 - mem_after_cat / mem_before_cat) * 100:.1f}% reduction)")

# Total memory AFTER all corrections
mem_after_all = df.memory_usage(deep=True).sum()
print(f"\nTotal memory usage AFTER all dtype corrections: {mem_after_all / 1024:.2f} KB "
      f"(vs {mem_before / 1024:.2f} KB before)")

In [ ]:
# All numeric columns
numeric_cols = df.select_dtypes(include="number").columns.tolist()
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")

# Descriptive statistics
print("\ndf.describe():")
print(df[numeric_cols].describe())

# Skewness per numeric column
skew_vals = df[numeric_cols].skew()
print("\nSkewness per numeric column (sorted by |skew|):")
print(skew_vals.sort_values(key=lambda s: s.abs(), ascending=False))

# Column with highest absolute skewness
most_skewed_col = skew_vals.abs().idxmax()
print(f"\nMost skewed column: '{most_skewed_col}' (skew = {skew_vals[most_skewed_col]:.3f})")

In [ ]:
# Outlier detection using IQR method for at least two numeric columns
iqr_cols = ["SalePrice", "Gr Liv Area"]

for col in iqr_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    n_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()

    print(f"Column: {col}")
    print(f"  Q1 = {Q1:.2f}, Q3 = {Q3:.2f}, IQR = {IQR:.2f}")
    print(f"  Lower bound = {lower_bound:.2f}, Upper bound = {upper_bound:.2f}")
    print(f"  Rows outside bounds: {n_outliers} ({n_outliers / len(df) * 100:.2f}%)")
    print()

# NOTE: outliers are NOT dropped here, only counted/documented as required.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(df.index, df["SalePrice"], color="#2563eb", linewidth=0.8)
plt.title("Sale Price Across Houses (by Row Index)")
plt.xlabel("Row Index")
plt.ylabel("Sale Price ($)")
plt.tight_layout()
plt.savefig("plot1_line.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
df.groupby("Neighborhood", observed=True)["SalePrice"].mean().sort_values().plot.bar(color="#f97316")
plt.title("Mean Sale Price by Neighborhood")
plt.xlabel("Neighborhood")
plt.ylabel("Mean Sale Price ($)")
plt.xticks(rotation=75)
plt.tight_layout()
plt.savefig("plot2_bar.png", dpi=150)
plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(9, 5))
sns.histplot(df[most_skewed_col].clip(upper=df[most_skewed_col].quantile(0.99)), bins=20, color="#16a34a")
plt.title(f"Distribution of {most_skewed_col} (most skewed, skew={skew_vals[most_skewed_col]:.2f})")
plt.xlabel(most_skewed_col)
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("plot3_histogram.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="Gr Liv Area", y="SalePrice", alpha=0.4, s=20, color="#7c3aed")
plt.title("Gr Liv Area vs SalePrice")
plt.xlabel("Above-Ground Living Area (sq ft)")
plt.ylabel("Sale Price ($)")
plt.tight_layout()
plt.savefig("plot4_scatter.png", dpi=150)
plt.show()

corr_val = df["Gr Liv Area"].corr(df["SalePrice"])
print(f"Correlation: {corr_val:.3f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x="Overall Qual", y="SalePrice", hue="Overall Qual", palette="viridis", legend=False)
plt.title("Sale Price by Overall Quality Rating")
plt.xlabel("Overall Quality (1-10)")
plt.ylabel("Sale Price ($)")
plt.tight_layout()
plt.savefig("plot5_boxplot.png", dpi=150)
plt.show()

In [ ]:
import numpy as np
import seaborn as sns

# Full correlation matrix (Order/PID excluded) - used to find the TOP correlated pair
numeric_df = df.select_dtypes(include=['number']).drop(columns=["Order", "PID"])
correlation_matrix = numeric_df.corr()

abs_corr_matrix = correlation_matrix.abs()
np.fill_diagonal(abs_corr_matrix.values, 0)
max_corr_pair = abs_corr_matrix.unstack().sort_values(ascending=False)

top_pair_cols = None
top_corr_value = None
for (col1, col2), value in max_corr_pair.items():
    if col1 != col2:
        top_pair_cols = (col1, col2)
        top_corr_value = value
        break

print(f"Pair of variables with the highest absolute correlation: {top_pair_cols}")
print(f"Highest absolute correlation value: {top_corr_value:.4f}")

# Smaller, readable heatmap with numbers shown (annot=True as required)
key_cols = ["SalePrice", "Overall Qual", "Gr Liv Area", "Garage Cars", "Garage Area",
            "Total Bsmt SF", "1st Flr SF", "Year Built", "Full Bath", "TotRms AbvGrd",
            "Year Remod/Add", "Fireplaces"]
corr_subset = df[key_cols].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr_subset, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Key Numeric Columns)")
plt.tight_layout()
plt.savefig("plot6_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Two most-skewed numeric columns (from Task 5)
top2_skew_cols = skew_vals.abs().sort_values(ascending=False).index[:2].tolist()
print(f"Two most-skewed numeric columns: {top2_skew_cols}")

for col in top2_skew_cols:
    mean_val = df[col].mean()
    median_val = df[col].median()
    nulls_before = df[col].isnull().sum()

    print(f"\n{col}:")
    print(f"  Mean (pre-imputation)   = {mean_val:.3f}")
    print(f"  Median (pre-imputation) = {median_val:.3f}")
    print(f"  Skew = {df[col].skew():.3f}")
    print(f"  Nulls before = {nulls_before}")

    # Both columns are positively skewed -> median is more representative
    df[col] = df[col].fillna(median_val)

    print(f"  -> Imputed with MEDIAN. Nulls after = {df[col].isnull().sum()}")

In [ ]:
import numpy as np
import pandas as pd

numeric_cols_clean = [c for c in df.select_dtypes(include="number").columns if c not in ["Order", "PID"]]

pearson_corr = df[numeric_cols_clean].corr(method="pearson")
spearman_corr = df[numeric_cols_clean].corr(method="spearman")

print("Pearson correlation matrix (first 5 rows/cols shown):")
print(pearson_corr.iloc[:5, :5].round(3))
print("\nSpearman correlation matrix (first 5 rows/cols shown):")
print(spearman_corr.iloc[:5, :5].round(3))

# |Spearman - Pearson| difference table, top 3 pairs
diff = (spearman_corr - pearson_corr).abs()
arr = diff.values.copy()
np.fill_diagonal(arr, 0)
diff_no_diag = pd.DataFrame(arr, index=diff.index, columns=diff.columns)

seen = set()
top3_pairs = []
for (a, b), val in diff_no_diag.unstack().sort_values(ascending=False).items():
    if (b, a) in seen:
        continue
    seen.add((a, b))
    top3_pairs.append((a, b, val, pearson_corr.loc[a, b], spearman_corr.loc[a, b]))
    if len(top3_pairs) == 3:
        break

diff_table = pd.DataFrame(top3_pairs, columns=["col_a", "col_b", "abs_diff", "pearson", "spearman"])
print("\nTop 3 pairs by |Spearman - Pearson|:")
print(diff_table)

In [ ]:
group_col = "Overall Qual"
value_col = "SalePrice"

agg_result = df.groupby(group_col, observed=True)[value_col].agg(["mean", "std", "count"])
agg_result = agg_result.sort_values("mean", ascending=False)
print(f"Grouped aggregation: {value_col} by {group_col}")
print(agg_result)

highest_mean_group = agg_result["mean"].idxmax()
highest_std_group = agg_result["std"].idxmax()
mean_ratio = agg_result["mean"].max() / agg_result["mean"].min()

print(f"\nGroup with highest mean: {highest_mean_group} ({agg_result['mean'].max():.2f})")
print(f"Group with highest std dev: {highest_std_group} ({agg_result['std'].max():.2f})")
print(f"Ratio of highest mean to lowest mean: {mean_ratio:.3f}")

In [ ]:
df.to_csv("cleaned_data.csv", index=False)
print(f"Saved cleaned_data.csv - final shape: {df.shape}")

from google.colab import files
files.download("cleaned_data.csv")